In [11]:
import sys
import torch
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [4]:
cwd = Path.cwd()
project_root = cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("done!")

done!


In [6]:
transformation = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [9]:
train_dataset = datasets.MNIST(
    root=project_root / "data",
    train=True,
    transform=transformation,
    download=True
)

In [10]:
test_dataset = datasets.MNIST(
    root=project_root / "data",
    train=False,
    transform=transformation,
    download=True
)

In [12]:
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [13]:
for img, label in train_dataloader:
    print(f"Label: {label}, shape: {label.shape}")
    print(f"img: {img}, shape: {img.shape}")
    break

Label: tensor([3, 7, 3, 9, 8, 5, 2, 9, 3, 8, 5, 4, 7, 8, 3, 2, 4, 6, 2, 5, 7, 5, 0, 3,
        3, 2, 0, 9, 0, 6, 1, 7, 7, 6, 6, 8, 3, 6, 1, 4, 0, 1, 1, 9, 4, 5, 8, 7,
        8, 1, 3, 7, 7, 3, 7, 0, 5, 3, 2, 1, 8, 1, 8, 2, 9, 8, 2, 1, 4, 2, 1, 6,
        5, 5, 4, 3, 6, 3, 3, 9, 2, 9, 5, 0, 3, 8, 1, 4, 6, 9, 7, 0, 3, 1, 9, 5,
        5, 3, 1, 9, 4, 1, 5, 1, 8, 5, 8, 5, 1, 5, 5, 6, 2, 9, 6, 7, 5, 7, 1, 4,
        6, 3, 5, 8, 6, 5, 0, 0]), shape: torch.Size([128])
img: tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]), shape: torch.Size([128, 784])


### Let's build the encoder

In [14]:
class Encoder(nn.Module):
    def __init__(self, in_channels, D):
        super().__init__()
        self.conv_layer1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=4*D,
            kernel_size=4,
            stride=2,
            padding=1
        )
        self.conv_layer2 = nn.Conv2d(
            in_channels=4*D,
            out_channels=2*D,
            kernel_size=4,
            stride=2,
            padding=1,
        )
        self.conv_layer3 = nn.Conv2d(
            in_channels=2*D,
            out_channels=D,
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.relu = nn.ReLU()
        
    def forward(self, X):
        X = self.conv_layer1(X)
        X = self.relu(X)
        X = self.conv_layer2(X)
        X = self.relu(X)
        X = self.conv_layer3(X)
        return X

### Building the CodeBook

In [ ]:
class CodeBook(nn.Module):
    def __init__(self, K, D):
        super().__init__()
        self.embedding = nn.Embedding(K, D)
        